# Project KIRA: ADV-002 Large-Scale Multi-Arm Stateful Adversarial Swarm Execution

**Experiment ID**: `ADV-002-LARGE`  
**Total Workload**: 15,000 Attempts across 3 Arms (5,000 per Arm: Adaptive Memory, Static Control, Memory-Disabled)  
**Compute Mode**: CPU Only (Standard Kaggle CPU Kernel)  
**Authoritative Baseline**: `run_tiny_s20260827_193f7897_40997ab`  
**Deadline Target**: < 60 minutes wall-clock ceiling (Projected: ~3-5 minutes)


In [ ]:
# 1. Setup Environment and Dependencies
!pip install -q polars scikit-learn lightgbm pytest

In [ ]:
# 2. Pre-flight: Verify Baseline 22/22 Artifacts & ADV-001 Memory Integrity
import hashlib
import sys
from pathlib import Path
sys.path.insert(0, 'src')

# Verify 22/22 baseline artifacts
!python3 tools/audit_authoritative_run.py run_tiny_s20260827_193f7897_40997ab

# Verify ADV-001 memory SHA-256
adv001_path = Path('research_runs/ADVANCED/ADV-001/attack_memory.jsonl')
if adv001_path.exists():
    h = hashlib.sha256()
    with open(adv001_path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    print(f'ADV-001 Attack Memory SHA-256: {h.hexdigest()}')
    assert h.hexdigest() == '7f37b59333a82d8fd04ab4e3582435cb798fa6e97cbcd1dc248122967e8087f7', 'ADV-001 hash mismatch!'
    print('PRE-FLIGHT ADV-001 INTEGRITY: PASS')
else:
    print('WARNING: ADV-001 memory not found at default path!')

In [ ]:
# 3. Execute Complete 3-Arm ADV-002 Large Swarm (15,000 Total Attempts)
# Arm A (Adaptive Memory: 5k) + Arm B (Static Control: 5k) + Arm C (Memory-Disabled: 5k)
!python3 -m mcdl.research.advanced.adv002.runner \
    --scale large \
    --mode all_arms \
    --output-dir research_runs/ADVANCED/ADV-002-LARGE \
    --deadline-sec 3000.0

In [ ]:
# 4. Post-Execution Regression Suite
!pytest tests/unit/research/test_adv002.py -v
!pytest tests/unit/research/test_adv001.py -v

In [ ]:
# 5. Package Evidence for Download
!zip -r adv002_large_evidence.zip research_runs/ADVANCED/ADV-002-LARGE/